In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data = fetch_california_housing()

df = pd.DataFrame(data=data.data, columns=data.feature_names)
df["MedHouseVal"] = data.target  # variável-alvo

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [ ]:
df["rooms_per_household"] = df["AveRooms"] / df["AveOccup"]
df["bedrooms_per_room"] = df["AveBedrms"] / df["AveRooms"]
df["population_per_household"] = df["Population"] / df["HouseAge"]
df["income_squared"] = df["MedInc"] ** 2


In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
r2_lr = r2_score(y_test, pred_lr)

print("Regressão Linear")
print("RMSE:", rmse_lr)
print("R²:", r2_lr)


Regressão Linear
RMSE: 0.6720303777639461
R²: 0.6553558193383138


In [ ]:
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
r2_xgb = r2_score(y_test, pred_xgb)

print("XGBoost")
print("RMSE:", rmse_xgb)
print("R²:", r2_xgb)


XGBoost
RMSE: 0.4551893496595862
R²: 0.8418834520489474


In [14]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=800,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)
pred_mlp = mlp.predict(X_test_scaled)

rmse_mlp = np.sqrt(mean_squared_error(y_test, pred_mlp))
r2_mlp = r2_score(y_test, pred_mlp)

print("Rede Neural (MLP)")
print("RMSE:", rmse_mlp)
print("R²:", r2_mlp)


Rede Neural (MLP)
RMSE: 0.5179072309056321
R²: 0.7953097816954747


In [15]:
results = pd.DataFrame({
    "Modelo": ["Regressão Linear", "XGBoost", "Rede Neural (MLP)"],
    "RMSE": [rmse_lr, rmse_xgb, rmse_mlp],
    "R²": [r2_lr, r2_xgb, r2_mlp]
})

results

,Modelo,RMSE,R²
0,Regressão Linear,0.672030,0.655356
1,XGBoost,0.455189,0.841883
2,Rede Neural (MLP),0.517907,0.795310
